# Estudio de Poses y Odometría (ROS2)
Este notebook analiza la calidad de la trayectoria del robot para detectar errores de sincronización o saltos en el TF.

In [ ]:
import os
import sys
sys.path.append('..')

import torch
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.transform import Rotation as R
from opensplat3d.data.ros2_reader import ROS2Reader
from rosbags.highlevel import AnyReader

In [ ]:
bag_path = "/home/jsm15/datasets/astra_lab/astra_lab_0.db3"
reader = ROS2Reader(
    bag_path,
    world_frame="odom",
    camera_frame="astra_camera_color_optical_frame",
    nth_frames=1 # Estudiamos TODOS los frames para ver el detalle
)

In [ ]:
times = []
positions = []
euler_angles = []
valid_mask = []

t0 = reader.frames[0]['timestamp'] if reader.frames else 0

for f in reader.frames:
    t = (f['timestamp'] - t0) / 1e9
    pose = f['pose']
    
    times.append(t)
    positions.append(pose[:3, 3])
    euler_angles.append(R.from_matrix(pose[:3, :3]).as_euler('xyz', degrees=True))
    valid_mask.append(True)

positions = np.array(positions)
euler_angles = np.array(euler_angles)
times = np.array(times)

## 1. Evolución Temporal (XYZ)

In [ ]:
plt.figure(figsize=(15, 8))
plt.subplot(3, 1, 1)
plt.plot(times, positions[:, 0], label='X', color='r')
plt.ylabel('X (m)')
plt.grid(True)
plt.subplot(3, 1, 2)
plt.plot(times, positions[:, 1], label='Y', color='g')
plt.ylabel('Y (m)')
plt.grid(True)
plt.subplot(3, 1, 3)
plt.plot(times, positions[:, 2], label='Z', color='b')
plt.ylabel('Z (m)')
plt.xlabel('Tiempo (s)')
plt.grid(True)
plt.suptitle("Posición de la cámara en el tiempo")
plt.show()

## 2. Orientación (Euler RPY)

In [ ]:
plt.figure(figsize=(15, 8))
plt.subplot(3, 1, 1)
plt.plot(times, euler_angles[:, 0], label='Roll', color='m')
plt.ylabel('Roll (deg)')
plt.grid(True)
plt.subplot(3, 1, 2)
plt.plot(times, euler_angles[:, 1], label='Pitch', color='c')
plt.ylabel('Pitch (deg)')
plt.grid(True)
plt.subplot(3, 1, 3)
plt.plot(times, euler_angles[:, 2], label='Yaw', color='y')
plt.ylabel('Yaw (deg)')
plt.xlabel('Tiempo (s)')
plt.grid(True)
plt.suptitle("Orientación de la cámara (Euler)")
plt.show()

## 3. Trayectoria 3D Birdseye

In [ ]:
plt.figure(figsize=(10, 10))
plt.plot(positions[:, 0], positions[:, 1], 'k-', alpha=0.3)
plt.scatter(positions[:, 0], positions[:, 1], c=times, cmap='viridis', s=10)
plt.colorbar(label='Tiempo (s)')
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.axis('equal')
plt.title("Trayectoria (Vista superior XY)")
plt.grid(True)
plt.show()